# mHC Reproduction Part 2: 300M Parameters on C4

**Goal**: Demonstrate that HC instability scales with depth at larger model sizes.

**Setup**:
- Model: ~300M params (dim=1024, n_heads=16)
- Depths: 24, 32, 48 layers
- Dataset: C4 subset (~50M tokens)
- Training: 10k steps per run
- Methods: HC and mHC at each depth (6 runs total)
- Hardware: A100 (Colab Pro)

**Runtime Estimate**: ~30-45 min per run, ~3-5 hours total

**Cost Estimate**: ~$1-3 (Colab Pro A100 rates)

## 1. Setup & Dependencies

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch datasets tiktoken matplotlib numpy tqdm

In [ ]:
# Mount Google Drive for saving results
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
import os
OUTPUT_DIR = '/content/drive/MyDrive/mhc_part2_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Results will be saved to: {OUTPUT_DIR}")

In [ ]:
import math
import json
import time
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, Callable, Literal

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from tqdm.auto import tqdm

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
@dataclass
class ModelConfig:
    """Model architecture configuration for ~300M params."""
    n_layers: int = 24
    hidden_dim: int = 1024
    n_heads: int = 16
    head_dim: int = 64
    ffn_multiplier: float = 4.0
    
    # Hyper-connection
    expansion_rate: int = 4
    hc_alpha: float = 0.01
    sinkhorn_iters: int = 20
    
    # Vocabulary (GPT-2 tokenizer)
    vocab_size: int = 50257
    max_seq_len: int = 512
    
    # Connection type
    connection_type: Literal["hc", "mhc"] = "mhc"
    
    # Dropout
    dropout: float = 0.1
    
    def __post_init__(self):
        assert self.hidden_dim % self.n_heads == 0
        self.head_dim = self.hidden_dim // self.n_heads


@dataclass
class TrainConfig:
    """Training configuration."""
    batch_size: int = 32  # Reduced for A100 memory
    context_length: int = 512
    max_steps: int = 10000
    eval_interval: int = 200
    eval_steps: int = 50
    log_interval: int = 50
    
    # Optimizer
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.95)
    grad_clip: float = 1.0
    warmup_steps: int = 500
    
    # Checkpointing
    save_interval: int = 2000
    
    # Reproducibility
    seed: int = 42


# Experiment configurations: depths to test
DEPTHS = [24, 32, 48]
METHODS = ["hc", "mhc"]

print("Experiment configurations:")
for depth in DEPTHS:
    for method in METHODS:
        print(f"  - {method.upper()} @ depth {depth}")

## 3. Data Loading (C4 Subset)

In [ ]:
import tiktoken
from datasets import load_dataset

# Use GPT-2 tokenizer
enc = tiktoken.get_encoding("gpt2")

def tokenize_text(text: str) -> list[int]:
    """Tokenize text using GPT-2 BPE."""
    return enc.encode(text, allowed_special={'<|endoftext|>'})


class C4Dataset(IterableDataset):
    """Streaming C4 dataset for language modeling."""
    
    def __init__(self, context_length: int, split: str = "train", max_tokens: int = 50_000_000):
        self.context_length = context_length
        self.split = split
        self.max_tokens = max_tokens
        self.enc = tiktoken.get_encoding("gpt2")
        
    def __iter__(self):
        # Load C4 streaming dataset
        dataset = load_dataset(
            "allenai/c4", 
            "en", 
            split=self.split,
            streaming=True,
            trust_remote_code=True
        )
        
        buffer = []
        tokens_seen = 0
        
        for example in dataset:
            # Tokenize
            tokens = self.enc.encode(example["text"], allowed_special={'<|endoftext|>'})
            tokens.append(self.enc.eot_token)  # Add end of text
            buffer.extend(tokens)
            tokens_seen += len(tokens)
            
            # Yield chunks when buffer is large enough
            while len(buffer) >= self.context_length + 1:
                x = torch.tensor(buffer[:self.context_length], dtype=torch.long)
                y = torch.tensor(buffer[1:self.context_length + 1], dtype=torch.long)
                yield x, y
                buffer = buffer[self.context_length:]
            
            # Stop after max_tokens
            if tokens_seen >= self.max_tokens:
                break


def create_dataloaders(config: TrainConfig):
    """Create train and validation dataloaders."""
    train_dataset = C4Dataset(
        context_length=config.context_length,
        split="train",
        max_tokens=50_000_000  # ~50M tokens for training
    )
    
    val_dataset = C4Dataset(
        context_length=config.context_length,
        split="validation",
        max_tokens=1_000_000  # ~1M tokens for validation
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        num_workers=2,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        num_workers=1,
    )
    
    return train_loader, val_loader


# Test data loading
print("Testing C4 data loading...")
test_config = TrainConfig()
test_dataset = C4Dataset(context_length=512, split="train", max_tokens=10000)
for i, (x, y) in enumerate(test_dataset):
    if i >= 3:
        break
    print(f"  Batch {i}: x.shape={x.shape}, y.shape={y.shape}")
    print(f"    Sample text: {enc.decode(x[:50].tolist())}...")
print("Data loading test passed!")

## 4. Model Architecture

In [ ]:
# ============================================================================
# Sinkhorn-Knopp Algorithm
# ============================================================================

def sinkhorn_knopp(H: torch.Tensor, t_max: int = 20, eps: float = 1e-8) -> torch.Tensor:
    """Project matrix onto doubly stochastic manifold."""
    P = torch.exp(H)
    for _ in range(t_max):
        P = P / (P.sum(dim=-1, keepdim=True) + eps)
        P = P / (P.sum(dim=-2, keepdim=True) + eps)
    return P


# ============================================================================
# Connection Types
# ============================================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.scale * x / rms


class HyperConnection(nn.Module):
    """Input-dependent Hyper-Connection (unconstrained H_res)."""
    
    def __init__(self, hidden_dim: int, expansion_rate: int = 4, alpha: float = 0.01, **kwargs):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n = expansion_rate
        self.uses_streams = True
        
        input_dim = self.n * hidden_dim
        
        self.rms_norm = RMSNorm(input_dim)
        
        # H_res projections
        self.theta_res = nn.Linear(input_dim, self.n * self.n, bias=False)
        self.b_res = nn.Parameter(torch.eye(self.n))
        self.alpha_res = nn.Parameter(torch.tensor(alpha))
        
        # H_pre projections
        self.theta_pre = nn.Linear(input_dim, self.n, bias=False)
        self.b_pre = nn.Parameter(torch.zeros(self.n))
        self.b_pre.data[0] = 1.0
        self.alpha_pre = nn.Parameter(torch.tensor(alpha))
        
        # H_post projections
        self.theta_post = nn.Linear(input_dim, self.n, bias=False)
        self.b_post = nn.Parameter(torch.zeros(self.n))
        self.b_post.data[0] = 1.0
        self.alpha_post = nn.Parameter(torch.tensor(alpha))
        
        # Initialize with small weights
        nn.init.normal_(self.theta_res.weight, std=0.01)
        nn.init.normal_(self.theta_pre.weight, std=0.01)
        nn.init.normal_(self.theta_post.weight, std=0.01)
        
        self._last_h_res = None

    def _compute_h_matrices(self, z: torch.Tensor):
        n, batch, seq, dim = z.shape
        z_pooled = z.mean(dim=2)
        z_flat = z_pooled.permute(1, 0, 2).reshape(batch, n * dim)
        x_norm = self.rms_norm(z_flat)
        
        H_res = self.alpha_res * torch.tanh(self.theta_res(x_norm)).reshape(batch, self.n, self.n) + self.b_res
        H_pre = self.alpha_pre * torch.tanh(self.theta_pre(x_norm)) + self.b_pre
        H_post = self.alpha_post * torch.tanh(self.theta_post(x_norm)) + self.b_post
        
        return H_res, H_pre, H_post

    def forward(self, z: torch.Tensor, sublayer: Callable) -> torch.Tensor:
        n, batch, seq, dim = z.shape
        H_res, H_pre, H_post = self._compute_h_matrices(z)
        self._last_h_res = H_res.mean(dim=0).detach()
        
        sublayer_input = torch.einsum("bn,nbsd->bsd", H_pre, z)
        sublayer_output = sublayer(sublayer_input)
        delta = sublayer_output.unsqueeze(0) * H_post.t().unsqueeze(-1).unsqueeze(-1)
        z_mixed = torch.einsum("bij,jbsd->ibsd", H_res, z)
        
        return z_mixed + delta

    def get_h_res(self):
        return self._last_h_res.clone() if self._last_h_res is not None else None


class ManifoldHyperConnection(nn.Module):
    """Manifold-Constrained Hyper-Connection (H_res projected to doubly stochastic)."""
    
    def __init__(self, hidden_dim: int, expansion_rate: int = 4, alpha: float = 0.01, sinkhorn_iters: int = 20, **kwargs):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n = expansion_rate
        self.sinkhorn_iters = sinkhorn_iters
        self.uses_streams = True
        
        input_dim = self.n * hidden_dim
        
        self.rms_norm = RMSNorm(input_dim)
        
        # H_res projections
        self.theta_res = nn.Linear(input_dim, self.n * self.n, bias=False)
        self.b_res = nn.Parameter(torch.eye(self.n) * 2.0)
        self.alpha_res = nn.Parameter(torch.tensor(alpha))
        
        # H_pre projections
        self.theta_pre = nn.Linear(input_dim, self.n, bias=False)
        self.b_pre = nn.Parameter(torch.zeros(self.n))
        self.b_pre.data[0] = 2.0
        self.alpha_pre = nn.Parameter(torch.tensor(alpha))
        
        # H_post projections
        self.theta_post = nn.Linear(input_dim, self.n, bias=False)
        self.b_post = nn.Parameter(torch.zeros(self.n))
        self.b_post.data[0] = 2.0
        self.alpha_post = nn.Parameter(torch.tensor(alpha))
        
        nn.init.normal_(self.theta_res.weight, std=0.01)
        nn.init.normal_(self.theta_pre.weight, std=0.01)
        nn.init.normal_(self.theta_post.weight, std=0.01)
        
        self._last_h_res = None

    def _compute_h_matrices(self, z: torch.Tensor):
        n, batch, seq, dim = z.shape
        z_pooled = z.mean(dim=2)
        z_flat = z_pooled.permute(1, 0, 2).reshape(batch, n * dim)
        x_norm = self.rms_norm(z_flat)
        
        H_res_raw = self.alpha_res * torch.tanh(self.theta_res(x_norm)).reshape(batch, self.n, self.n) + self.b_res
        H_res = sinkhorn_knopp(H_res_raw, t_max=self.sinkhorn_iters)
        
        H_pre_raw = self.alpha_pre * torch.tanh(self.theta_pre(x_norm)) + self.b_pre
        H_pre = torch.sigmoid(H_pre_raw)
        
        H_post_raw = self.alpha_post * torch.tanh(self.theta_post(x_norm)) + self.b_post
        H_post = 2.0 * torch.sigmoid(H_post_raw)
        
        return H_res, H_pre, H_post

    def forward(self, z: torch.Tensor, sublayer: Callable) -> torch.Tensor:
        n, batch, seq, dim = z.shape
        H_res, H_pre, H_post = self._compute_h_matrices(z)
        self._last_h_res = H_res.mean(dim=0).detach()
        
        sublayer_input = torch.einsum("bn,nbsd->bsd", H_pre, z)
        sublayer_output = sublayer(sublayer_input)
        delta = sublayer_output.unsqueeze(0) * H_post.t().unsqueeze(-1).unsqueeze(-1)
        z_mixed = torch.einsum("bij,jbsd->ibsd", H_res, z)
        
        return z_mixed + delta

    def get_h_res(self):
        return self._last_h_res.clone() if self._last_h_res is not None else None


def get_connection_class(connection_type: str):
    return {"hc": HyperConnection, "mhc": ManifoldHyperConnection}[connection_type]

In [ ]:
# ============================================================================
# Transformer Components
# ============================================================================

class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention."""
    
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.head_dim
        self.hidden_dim = config.hidden_dim
        
        self.qkv = nn.Linear(config.hidden_dim, 3 * config.hidden_dim, bias=False)
        self.out_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
        self.dropout = nn.Dropout(config.dropout)
        
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len))
        self.register_buffer("mask", mask.view(1, 1, config.max_seq_len, config.max_seq_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        scale = 1.0 / math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        y = self.dropout(y)
        
        return y


class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    
    def __init__(self, config: ModelConfig):
        super().__init__()
        hidden_dim = config.hidden_dim
        ffn_dim = int(hidden_dim * config.ffn_multiplier)
        
        self.fc1 = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.fc2 = nn.Linear(ffn_dim, hidden_dim, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer block with configurable connection type."""
    
    def __init__(self, config: ModelConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        
        self.ln1 = nn.LayerNorm(config.hidden_dim)
        self.ln2 = nn.LayerNorm(config.hidden_dim)
        self.attn = CausalSelfAttention(config)
        self.ffn = FeedForward(config)
        
        ConnectionClass = get_connection_class(config.connection_type)
        self.conn_attn = ConnectionClass(
            hidden_dim=config.hidden_dim,
            expansion_rate=config.expansion_rate,
            alpha=config.hc_alpha,
            sinkhorn_iters=config.sinkhorn_iters,
        )
        self.conn_ffn = ConnectionClass(
            hidden_dim=config.hidden_dim,
            expansion_rate=config.expansion_rate,
            alpha=config.hc_alpha,
            sinkhorn_iters=config.sinkhorn_iters,
        )
        self.uses_streams = self.conn_attn.uses_streams

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conn_attn(x, lambda z: self.attn(self.ln1(z)))
        x = self.conn_ffn(x, lambda z: self.ffn(self.ln2(z)))
        return x

    def get_h_res_matrices(self):
        return {"attn": self.conn_attn.get_h_res(), "ffn": self.conn_ffn.get_h_res()}


class GPT(nn.Module):
    """GPT model with HC/mHC connections."""
    
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        self.tok_emb = nn.Embedding(config.vocab_size, config.hidden_dim)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.hidden_dim)
        self.drop = nn.Dropout(config.dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(config, i) for i in range(config.n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(config.hidden_dim)
        self.lm_head = nn.Linear(config.hidden_dim, config.vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight  # Weight tying
        
        self.apply(self._init_weights)
        self.uses_streams = True
        self.n = config.expansion_rate

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)

    def _expand_streams(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(0).expand(self.n, -1, -1, -1).clone()

    def _collapse_streams(self, z: torch.Tensor) -> torch.Tensor:
        return z[0]

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None):
        B, T = idx.shape
        device = idx.device
        
        tok_emb = self.tok_emb(idx)
        pos = torch.arange(0, T, device=device).unsqueeze(0)
        pos_emb = self.pos_emb(pos)
        x = self.drop(tok_emb + pos_emb)
        
        x = self._expand_streams(x)
        for block in self.blocks:
            x = block(x)
        x = self._collapse_streams(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        
        return logits, loss

    def get_all_h_res_matrices(self):
        return [block.get_h_res_matrices() for block in self.blocks]

    def get_composite_h_res(self):
        matrices = self.get_all_h_res_matrices()
        if matrices[0]["attn"] is None:
            return None
        n = matrices[0]["attn"].shape[0]
        composite = torch.eye(n, device=matrices[0]["attn"].device)
        for layer_matrices in matrices:
            if layer_matrices["attn"] is not None:
                composite = composite @ layer_matrices["attn"]
            if layer_matrices["ffn"] is not None:
                composite = composite @ layer_matrices["ffn"]
        return composite

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Test model creation and parameter count
print("Testing model creation...")
for depth in DEPTHS:
    config = ModelConfig(n_layers=depth, connection_type="mhc")
    model = GPT(config)
    n_params = model.count_parameters()
    print(f"  Depth {depth}: {n_params:,} params ({n_params/1e6:.1f}M)")
    del model
    torch.cuda.empty_cache()

## 5. Metrics and Training Loop

In [ ]:
def compute_amax_gain(h_res: torch.Tensor) -> float:
    """Compute Amax Gain Magnitude."""
    if h_res is None:
        return 0.0
    row_sums = h_res.abs().sum(dim=-1)
    col_sums = h_res.abs().sum(dim=-2)
    return max(row_sums.max().item(), col_sums.max().item())


def compute_gradient_norm(model: nn.Module) -> float:
    """Compute total gradient norm."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm ** 0.5


def compute_layer_gradient_norms(model: GPT) -> list[dict]:
    """Compute per-layer gradient norms."""
    layer_norms = []
    for i, block in enumerate(model.blocks):
        norms = {"layer": i}
        
        attn_norm = sum(p.grad.data.norm(2).item() ** 2 for p in block.attn.parameters() if p.grad is not None) ** 0.5
        ffn_norm = sum(p.grad.data.norm(2).item() ** 2 for p in block.ffn.parameters() if p.grad is not None) ** 0.5
        conn_attn_norm = sum(p.grad.data.norm(2).item() ** 2 for p in block.conn_attn.parameters() if p.grad is not None) ** 0.5
        conn_ffn_norm = sum(p.grad.data.norm(2).item() ** 2 for p in block.conn_ffn.parameters() if p.grad is not None) ** 0.5
        
        norms["attn"] = attn_norm
        norms["ffn"] = ffn_norm
        norms["conn_attn"] = conn_attn_norm
        norms["conn_ffn"] = conn_ffn_norm
        norms["total"] = (attn_norm**2 + ffn_norm**2 + conn_attn_norm**2 + conn_ffn_norm**2) ** 0.5
        
        layer_norms.append(norms)
    return layer_norms


def collect_metrics(model: GPT, loss: float, step: int) -> dict:
    """Collect all metrics for a training step."""
    metrics = {
        "step": step,
        "loss": loss,
        "grad_norm": compute_gradient_norm(model),
        "layer_grad_norms": compute_layer_gradient_norms(model),
    }
    
    h_res_all = model.get_all_h_res_matrices()
    if h_res_all[0]["attn"] is not None:
        layer_amax = []
        for i, layer_h in enumerate(h_res_all):
            layer_amax.append({
                "layer": i,
                "attn": compute_amax_gain(layer_h["attn"]),
                "ffn": compute_amax_gain(layer_h["ffn"]),
            })
        metrics["layer_amax"] = layer_amax
        metrics["composite_amax"] = compute_amax_gain(model.get_composite_h_res())
    
    return metrics


def get_lr(step: int, config: TrainConfig) -> float:
    """Learning rate with warmup and cosine decay."""
    if step < config.warmup_steps:
        return config.learning_rate * step / config.warmup_steps
    decay_ratio = (step - config.warmup_steps) / (config.max_steps - config.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config.learning_rate * 0.1 + coeff * config.learning_rate * 0.9


@torch.no_grad()
def evaluate(model: GPT, val_loader, device: str, max_steps: int = 50) -> float:
    """Compute validation loss."""
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= max_steps:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses) if losses else 0.0

In [ ]:
def train_model(
    connection_type: str,
    depth: int,
    output_dir: str,
    train_config: TrainConfig = None,
) -> dict:
    """Train a single model configuration."""
    if train_config is None:
        train_config = TrainConfig()
    
    run_dir = Path(output_dir) / f"{connection_type}_depth_{depth}"
    run_dir.mkdir(parents=True, exist_ok=True)
    
    # Config
    model_config = ModelConfig(
        n_layers=depth,
        connection_type=connection_type,
    )
    
    # Set seed
    torch.manual_seed(train_config.seed)
    
    # Data
    train_loader, val_loader = create_dataloaders(train_config)
    
    # Model
    model = GPT(model_config).to(device)
    n_params = model.count_parameters()
    print(f"\n{'='*60}")
    print(f"Training {connection_type.upper()} @ depth {depth}")
    print(f"Parameters: {n_params:,} ({n_params/1e6:.1f}M)")
    print(f"{'='*60}")
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=train_config.learning_rate,
        betas=train_config.betas,
        weight_decay=train_config.weight_decay,
    )
    
    # Training loop
    model.train()
    history = []
    train_iter = iter(train_loader)
    best_val_loss = float("inf")
    start_time = time.time()
    
    pbar = tqdm(range(train_config.max_steps), desc=f"{connection_type.upper()} D={depth}")
    
    for step in pbar:
        # Get batch
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)
        
        x, y = x.to(device), y.to(device)
        
        # Update learning rate
        lr = get_lr(step, train_config)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        
        # Forward
        _, loss = model(x, y)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        if train_config.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.grad_clip)
        
        # Collect metrics BEFORE optimizer step
        if step % train_config.log_interval == 0:
            metrics = collect_metrics(model, loss.item(), step)
            metrics["lr"] = lr
            history.append(metrics)
            
            # Update progress bar
            pbar_desc = f"{connection_type.upper()} D={depth} | Loss: {loss.item():.4f}"
            if "composite_amax" in metrics:
                pbar_desc += f" | Amax: {metrics['composite_amax']:.2f}"
            pbar.set_description(pbar_desc)
        
        # Optimizer step
        optimizer.step()
        
        # Evaluation
        if step > 0 and step % train_config.eval_interval == 0:
            val_loss = evaluate(model, val_loader, device, train_config.eval_steps)
            if history:
                history[-1]["val_loss"] = val_loss
            print(f"  Step {step}: val_loss = {val_loss:.4f}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), run_dir / "best_model.pt")
        
        # Checkpointing
        if step > 0 and step % train_config.save_interval == 0:
            torch.save({
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
            }, run_dir / f"checkpoint_{step}.pt")
    
    # Final evaluation
    val_loss = evaluate(model, val_loader, device, train_config.eval_steps)
    elapsed = time.time() - start_time
    
    print(f"\nTraining complete!")
    print(f"  Final val loss: {val_loss:.4f}")
    print(f"  Best val loss: {best_val_loss:.4f}")
    print(f"  Time: {elapsed/60:.1f} minutes")
    
    # Save final model and history
    torch.save(model.state_dict(), run_dir / "final_model.pt")
    
    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    
    with open(run_dir / "config.json", "w") as f:
        json.dump({
            "model": asdict(model_config),
            "train": asdict(train_config),
            "results": {
                "final_val_loss": val_loss,
                "best_val_loss": best_val_loss,
                "training_time_minutes": elapsed / 60,
            }
        }, f, indent=2)
    
    # Cleanup
    del model, optimizer
    torch.cuda.empty_cache()
    
    return {"history": history, "best_val_loss": best_val_loss, "time_minutes": elapsed / 60}

## 6. Run All Experiments

In [ ]:
# Run all 6 experiments: HC and mHC at depths 24, 32, 48
results = {}

train_config = TrainConfig(
    batch_size=32,  # Adjust based on GPU memory
    max_steps=10000,
    eval_interval=200,
    log_interval=50,
)

total_start = time.time()

for depth in DEPTHS:
    for method in METHODS:
        key = f"{method}_depth_{depth}"
        print(f"\n\n{'#'*70}")
        print(f"# Starting: {key}")
        print(f"{'#'*70}")
        
        results[key] = train_model(
            connection_type=method,
            depth=depth,
            output_dir=OUTPUT_DIR,
            train_config=train_config,
        )

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"ALL EXPERIMENTS COMPLETE")
print(f"Total time: {total_time/3600:.2f} hours")
print(f"{'='*70}")

## 7. Visualization

In [ ]:
# Plotting style (matching Part 1)
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

COLORS = {
    "hc": "#E63946",   # Red
    "mhc": "#2A9D8F",  # Teal
}

LABELS = {
    "hc": "HC",
    "mhc": "mHC",
}


def load_history(path: Path):
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)


def smooth(values, window=20):
    arr = np.array(values)
    if len(arr) < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode="valid")

In [ ]:
def plot_part2_results(output_dir: str):
    """Generate Part 2 publication figures."""
    output_path = Path(output_dir)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    fig.suptitle("mHC Reproduction Part 2: 300M Parameters on C4", 
                 fontsize=16, fontweight="bold", y=1.02)
    
    # Load all histories
    histories = {}
    for depth in DEPTHS:
        for method in METHODS:
            key = f"{method}_depth_{depth}"
            histories[key] = load_history(output_path / key / "history.json")
    
    # Panel (a): Val Loss vs Depth
    ax = axes[0, 0]
    for method in METHODS:
        final_losses = []
        for depth in DEPTHS:
            h = histories.get(f"{method}_depth_{depth}")
            if h:
                val_losses = [s["val_loss"] for s in h if "val_loss" in s]
                final_losses.append(val_losses[-1] if val_losses else None)
            else:
                final_losses.append(None)
        
        valid_depths = [d for d, l in zip(DEPTHS, final_losses) if l is not None]
        valid_losses = [l for l in final_losses if l is not None]
        
        if valid_depths:
            ax.plot(valid_depths, valid_losses, "o-", color=COLORS[method], 
                   lw=2, ms=8, label=LABELS[method])
    
    ax.set_xlabel("Depth (layers)")
    ax.set_ylabel("Final Validation Loss")
    ax.set_title("(a) Val Loss vs Depth")
    ax.set_xticks(DEPTHS)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Panel (b): Max Amax vs Depth
    ax = axes[0, 1]
    for method in METHODS:
        max_amax_values = []
        for depth in DEPTHS:
            h = histories.get(f"{method}_depth_{depth}")
            if h:
                amax = [s["composite_amax"] for s in h if "composite_amax" in s]
                max_amax_values.append(max(amax) if amax else None)
            else:
                max_amax_values.append(None)
        
        valid_depths = [d for d, a in zip(DEPTHS, max_amax_values) if a is not None]
        valid_amax = [a for a in max_amax_values if a is not None]
        
        if valid_depths:
            ax.plot(valid_depths, valid_amax, "o-", color=COLORS[method], 
                   lw=2, ms=8, label=LABELS[method])
    
    ax.axhline(y=1.0, color="gray", linestyle="--", lw=2, alpha=0.8, label="Stability bound")
    ax.set_xlabel("Depth (layers)")
    ax.set_ylabel("Max Composite Amax")
    ax.set_title("(b) Max Amax vs Depth")
    ax.set_xticks(DEPTHS)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left")
    
    # Panel (c): Gradient Norm vs Depth (NEW)
    ax = axes[0, 2]
    for method in METHODS:
        max_grad_values = []
        for depth in DEPTHS:
            h = histories.get(f"{method}_depth_{depth}")
            if h:
                grads = [s["grad_norm"] for s in h if "grad_norm" in s]
                max_grad_values.append(max(grads) if grads else None)
            else:
                max_grad_values.append(None)
        
        valid_depths = [d for d, g in zip(DEPTHS, max_grad_values) if g is not None]
        valid_grads = [g for g in max_grad_values if g is not None]
        
        if valid_depths:
            ax.plot(valid_depths, valid_grads, "o-", color=COLORS[method], 
                   lw=2, ms=8, label=LABELS[method])
    
    ax.set_xlabel("Depth (layers)")
    ax.set_ylabel("Max Gradient Norm")
    ax.set_title("(c) Max Gradient Norm vs Depth")
    ax.set_xticks(DEPTHS)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Panel (d): Training Loss at Depth 48
    ax = axes[1, 0]
    for method in METHODS:
        h = histories.get(f"{method}_depth_48")
        if h:
            steps = [s["step"] for s in h]
            losses = [s["loss"] for s in h]
            smoothed = smooth(losses)
            ax.plot(steps[19:], smoothed, color=COLORS[method], lw=2, label=LABELS[method])
    
    ax.set_xlabel("Step")
    ax.set_ylabel("Training Loss")
    ax.set_title("(d) Training Loss at Depth 48")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Panel (e): Amax Evolution at Depth 48
    ax = axes[1, 1]
    for method in METHODS:
        h = histories.get(f"{method}_depth_48")
        if h:
            steps = [s["step"] for s in h if "composite_amax" in s]
            amax = [s["composite_amax"] for s in h if "composite_amax" in s]
            smoothed = smooth(amax)
            ax.plot(steps[19:], smoothed, color=COLORS[method], lw=2, label=LABELS[method])
    
    ax.axhline(y=1.0, color="gray", linestyle="--", lw=2, alpha=0.8, label="Stability bound")
    ax.set_xlabel("Step")
    ax.set_ylabel("Composite Amax")
    ax.set_title("(e) Amax Evolution at Depth 48")
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Panel (f): Gradient Norm Evolution at Depth 48 (NEW)
    ax = axes[1, 2]
    for method in METHODS:
        h = histories.get(f"{method}_depth_48")
        if h:
            steps = [s["step"] for s in h if "grad_norm" in s]
            grads = [s["grad_norm"] for s in h if "grad_norm" in s]
            smoothed = smooth(grads)
            ax.plot(steps[19:], smoothed, color=COLORS[method], lw=2, label=LABELS[method])
    
    ax.set_xlabel("Step")
    ax.set_ylabel("Gradient Norm")
    ax.set_title("(f) Gradient Norm Evolution at Depth 48")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path / "part2_results.png")
    plt.show()
    print(f"Saved part2_results.png to {output_path}")


# Generate figures
plot_part2_results(OUTPUT_DIR)

In [ ]:
def plot_depth_comparison(output_dir: str):
    """Plot training curves across all depths."""
    output_path = Path(output_dir)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    fig.suptitle("Training Dynamics by Depth", fontsize=16, fontweight="bold", y=1.02)
    
    # Load histories
    histories = {}
    for depth in DEPTHS:
        for method in METHODS:
            key = f"{method}_depth_{depth}"
            histories[key] = load_history(output_path / key / "history.json")
    
    # Depth gradient colors
    depth_colors = {24: 0.3, 32: 0.6, 48: 0.9}
    
    for i, depth in enumerate(DEPTHS):
        # Training loss
        ax = axes[0, i]
        for method in METHODS:
            h = histories.get(f"{method}_depth_{depth}")
            if h:
                steps = [s["step"] for s in h]
                losses = [s["loss"] for s in h]
                smoothed = smooth(losses)
                ax.plot(steps[19:], smoothed, color=COLORS[method], lw=2, label=LABELS[method])
        
        ax.set_xlabel("Step")
        ax.set_ylabel("Training Loss")
        ax.set_title(f"Depth {depth}: Training Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Amax evolution
        ax = axes[1, i]
        for method in METHODS:
            h = histories.get(f"{method}_depth_{depth}")
            if h:
                steps = [s["step"] for s in h if "composite_amax" in s]
                amax = [s["composite_amax"] for s in h if "composite_amax" in s]
                smoothed = smooth(amax)
                ax.plot(steps[19:], smoothed, color=COLORS[method], lw=2, label=LABELS[method])
        
        ax.axhline(y=1.0, color="gray", linestyle="--", lw=2, alpha=0.8)
        ax.set_xlabel("Step")
        ax.set_ylabel("Composite Amax")
        ax.set_title(f"Depth {depth}: Amax Evolution")
        ax.set_ylim(bottom=0)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path / "depth_comparison.png")
    plt.show()
    print(f"Saved depth_comparison.png to {output_path}")


plot_depth_comparison(OUTPUT_DIR)

In [ ]:
def print_results_summary(output_dir: str):
    """Print summary table of results."""
    output_path = Path(output_dir)
    
    print("\n" + "="*80)
    print("PART 2 RESULTS SUMMARY (300M Parameters, C4 Dataset)")
    print("="*80)
    print(f"\n{'Method':<8} {'Depth':<8} {'Final Val Loss':<16} {'Max Amax':<12} {'Max Grad':<12} {'Time (min)':<12}")
    print("-"*80)
    
    for depth in DEPTHS:
        for method in METHODS:
            key = f"{method}_depth_{depth}"
            config_path = output_path / key / "config.json"
            history_path = output_path / key / "history.json"
            
            if config_path.exists():
                with open(config_path, "r") as f:
                    config = json.load(f)
                
                h = load_history(history_path)
                if h:
                    val_losses = [s["val_loss"] for s in h if "val_loss" in s]
                    amax = [s["composite_amax"] for s in h if "composite_amax" in s]
                    grads = [s["grad_norm"] for s in h if "grad_norm" in s]
                    
                    final_loss = val_losses[-1] if val_losses else "N/A"
                    max_amax = max(amax) if amax else "N/A"
                    max_grad = max(grads) if grads else "N/A"
                    train_time = config.get("results", {}).get("training_time_minutes", "N/A")
                    
                    print(f"{method.upper():<8} {depth:<8} {final_loss:<16.4f} {max_amax:<12.2f} {max_grad:<12.2f} {train_time:<12.1f}")
    
    print("="*80)


print_results_summary(OUTPUT_DIR)

## 8. Download Results

All results are saved to Google Drive. You can also download them directly.

In [ ]:
# Create a zip file of all results for easy download
import shutil

zip_path = "/content/mhc_part2_results.zip"
shutil.make_archive("/content/mhc_part2_results", "zip", OUTPUT_DIR)
print(f"Results archived to: {zip_path}")

# Download link
from google.colab import files
files.download(zip_path)